In [11]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import gee
from statsmodels.formula.api import mixedlm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm


In [9]:
# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
FILE_PATH = 'df_long_sucio.csv'  # Cambia esto si usas otro archivo

print(f"--- ANALIZANDO ARCHIVO: {FILE_PATH} ---")

# ==========================================
# 2. CARGA Y LIMPIEZA
# ==========================================
try:
    df = pd.read_csv(FILE_PATH)
except FileNotFoundError:
    raise FileNotFoundError("Error: Archivo no encontrado.")

# Filtros rigurosos: Sin atención, solo primer bloque (Entre-sujetos)
df_clean = df[
    (df['Dilema'] != 'Atencion') & 
    (df['Orden_1'] == 1) & 
    (df['Gap_Size'].notnull())
].copy()

if 'Dilema' in df_clean.columns:
    df_clean.rename(columns={'Dilema': 'Tratamiento'}, inplace=True)

print(f"Sujetos válidos: {df_clean['ID_Sujeto'].nunique()}")

# ==========================================
# TABLA 1: ANOVA DE PROMEDIOS (LA QUE NECESITAS)
# ==========================================
print("\n" + "-"*50)
print("TABLA 1: ANOVA DE PROMEDIOS (Sujetos)")
print("")
print("-"*50)

# 1. Colapsar: Promedio por sujeto
df_subject_means = df_clean.groupby(['ID_Sujeto', 'Tratamiento'])['Mantiene'].mean().reset_index()

# 2. ANOVA Simple
model_anova = ols('Mantiene ~ C(Tratamiento)', data=df_subject_means).fit()
anova_table = anova_lm(model_anova, typ=2)

print(anova_table)
print("\n--- Medias de condiciones' ---")
print(df_subject_means.groupby('Tratamiento')['Mantiene'].mean())

# ==========================================
# TABLA 2: MODELO MIXTO (WALD TESTS)
# ==========================================

print("\n" + "-"*50)
print("TABLA 2: MODELO MIXTO (WALD TESTS)")
print("")
print("-"*50)

formula = "Mantiene ~ C(Tratamiento) * C(Gap_Size)"
try:
    model_mixed = mixedlm(formula, df_clean, groups=df_clean["ID_Sujeto"])
    res_mixed = model_mixed.fit()
    
    # Wald Tests Manuales
    wt_trat = res_mixed.wald_test(['C(Tratamiento)[T.Bloque_SIN] = 0', 'C(Tratamiento)[T.Dist] = 0'])
    wt_gap = res_mixed.wald_test([
        'C(Gap_Size)[T.1000.0] = 0', 'C(Gap_Size)[T.1200.0] = 0',
        'C(Gap_Size)[T.2000.0] = 0', 'C(Gap_Size)[T.2400.0] = 0'
    ])
    terms_int = [x for x in res_mixed.params.index if ':' in x]
    wt_int = res_mixed.wald_test(terms_int)
    
    print(f"{'FUENTE':<20} | {'CHI2':<10} | {'P-VALOR':<10}")
    print("-" * 45)
    print(f"{'Tratamiento':<20} | {wt_trat.statistic.item():<10.2f} | {wt_trat.pvalue:<10.4f}")
    print(f"{'Gap (Costo)':<20} | {wt_gap.statistic.item():<10.2f} | {wt_gap.pvalue:<10.4f}")
    print(f"{'Interacción':<20} | {wt_int.statistic.item():<10.2f} | {wt_int.pvalue:<10.4f}")

except Exception as e:
    print(f"Error en Modelo Mixto: {e}")

# ==========================================
# TABLA 3: GEE (ROBUSTO)
# ==========================================
print("\n" + "-"*50)
print("TABLA 3: GEE (ROBUSTO - Alternativa)")
print("-"*50)

try:
    fam = sm.families.Gaussian()
    ind = sm.cov_struct.Exchangeable()
    model_gee = gee(formula, "ID_Sujeto", df_clean, cov_struct=ind, family=fam)
    res_gee = model_gee.fit()
    
    wt_trat_g = res_gee.wald_test(['C(Tratamiento)[T.Bloque_SIN] = 0', 'C(Tratamiento)[T.Dist] = 0'])
    wt_gap_g = res_gee.wald_test([
        'C(Gap_Size)[T.1000.0] = 0', 'C(Gap_Size)[T.1200.0] = 0',
        'C(Gap_Size)[T.2000.0] = 0', 'C(Gap_Size)[T.2400.0] = 0'
    ])
    terms_int_g = [x for x in res_gee.params.index if ':' in x]
    wt_int_g = res_gee.wald_test(terms_int_g)
    
    print(f"{'FUENTE':<20} | {'CHI2':<10} | {'P-VALOR':<10}")
    print("-" * 45)
    print(f"{'Tratamiento':<20} | {wt_trat_g.statistic.item():<10.2f} | {wt_trat_g.pvalue:<10.4f}")
    print(f"{'Gap (Costo)':<20} | {wt_gap_g.statistic.item():<10.2f} | {wt_gap_g.pvalue:<10.4f}")
    print(f"{'Interacción':<20} | {wt_int_g.statistic.item():<10.2f} | {wt_int_g.pvalue:<10.4f}")

except Exception as e:
    print(f"Error en GEE: {e}")

--- ANALIZANDO ARCHIVO: df_long_sucio.csv ---
Sujetos válidos: 162

--------------------------------------------------
TABLA 1: ANOVA DE PROMEDIOS (Sujetos)

--------------------------------------------------
                   sum_sq     df         F    PR(>F)
C(Tratamiento)   0.446088    2.0  3.076008  0.048899
Residual        11.529221  159.0       NaN       NaN

--- Medias de condiciones' ---
Tratamiento
Bloque_CON    0.326389
Bloque_SIN    0.448485
Dist          0.350282
Name: Mantiene, dtype: float64

--------------------------------------------------
TABLA 2: MODELO MIXTO (WALD TESTS)

--------------------------------------------------
FUENTE               | CHI2       | P-VALOR   
---------------------------------------------
Tratamiento          | 5.05       | 0.0802    
Gap (Costo)          | 40.41      | 0.0000    
Interacción          | 3.36       | 0.9101    

--------------------------------------------------
TABLA 3: GEE (ROBUSTO - Alternativa)
--------------------------

C:\Users\felip\anaconda3\lib\site-packages\statsmodels\base\model.py:1914: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
C:\Users\felip\anaconda3\lib\site-packages\statsmodels\base\model.py:1914: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
